# `pyinform` — Information Dynamics in Discrete Time Series

> **Docs**: https://elife-asu.github.io/PyInform

**`pyinform`** quantifies *temporal* information structure in discrete time series.
Its key measures are **asymmetric** — they detect the direction of information flow.

### What we compute
| Measure | Symbol | Interpretation |
|---------|--------|---------------|
| Transfer Entropy | $TE_{X \to Y}$ | How much $X$'s past reduces uncertainty about $Y$'s next state |
| Active Information Storage | $AIS(Y)$ | How much $Y$'s own past predicts its next state |

$TE_{X \to Y} \neq TE_{Y \to X}$ in general — this **asymmetry** reveals causal direction.

### Datasets
* **Artificial time series** — $Y$ is an autoregressive process driven by $X$ ($X \to Y$ is the true causal link).
* **Real data** — UCI Wine: we discretise two top features and compute TE in both directions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import KBinsDiscretizer

from pyinform.transferentropy import transfer_entropy
from pyinform.activeinfo import active_info

SEED = 42
rng = np.random.default_rng(SEED)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
print("Imports OK.")


## 1. Construct a Causal Time Series

We build a simple binary process where $Y$ is mostly autoregressive but occasionally
copies $X$'s previous state, creating a genuine $X \to Y$ link.

$$
Y_t = \begin{cases}
Y_{t-1} & \text{with prob. } p_{\mathrm{self}} = 0.9 \\
X_{t-1} & \text{otherwise}
\end{cases}
$$


In [ ]:
T = 500

# X: independent fair coin (no memory, no causal source)
x = rng.integers(0, 2, size=T)

# Y: autoregressive + driven by X
y = np.zeros(T, dtype=int)
y[0] = rng.integers(0, 2)
p_self = 0.9

for t in range(1, T):
    if rng.random() < p_self:
        y[t] = y[t-1]        # keep own state
    else:
        y[t] = x[t-1]        # copy X

print(f"Series length: {T}")
print(f"X sample: {x[:20].tolist()}")
print(f"Y sample: {y[:20].tolist()}")
print(f"\nTrue causal structure: X → Y")


## 2. Transfer Entropy and Active Information Storage

In [ ]:
# Transfer Entropy (k=1: use one time step of history)
te_xy = transfer_entropy(x.tolist(), y.tolist(), k=1)
te_yx = transfer_entropy(y.tolist(), x.tolist(), k=1)

print("Transfer Entropy")
print(f"  TE(X→Y) = {te_xy:.4f} bits   ← should be HIGH (true causal link)")
print(f"  TE(Y→X) = {te_yx:.4f} bits   ← should be LOW  (no reverse link)")
print(f"  ΔTE = TE(X→Y) - TE(Y→X) = {te_xy - te_yx:.4f} bits")

# Active Information Storage
ais_x = active_info(x.tolist(), k=1)
ais_y = active_info(y.tolist(), k=1)

print(f"\nActive Information Storage")
print(f"  AIS(X) = {ais_x:.4f} bits   ← X is i.i.d.; low self-predictability")
print(f"  AIS(Y) = {ais_y:.4f} bits   ← Y is autoregressive; high self-predictability")


## 3. Visualisation — Artificial Data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(['$TE_{X\\to Y}$', '$TE_{Y\\to X}$'], [te_xy, te_yx],
            color=[sns.color_palette()[0], sns.color_palette()[3]])
axes[0].set_ylabel('Transfer Entropy (bits)')
axes[0].set_title('Directed information flow\n(X→Y is the true causal link)',
                  fontweight='bold')

axes[1].bar(['$AIS(X)$', '$AIS(Y)$'], [ais_x, ais_y],
            color=[sns.color_palette()[2], sns.color_palette()[1]])
axes[1].set_ylabel('Active Information Storage (bits)')
axes[1].set_title('Self-predictability / memory\n(Y is AR; X is i.i.d.)',
                  fontweight='bold')

plt.tight_layout()
plt.show()


## 4. Real Data — UCI Wine Dataset

We discretise the top-2 features (by MI with class label) and compute TE in both directions.
This checks whether the features have a detectable temporal ordering 

> **Note**: the Wine dataset is not a time series, thus the dynamical interpretation is meaningless. Wine samples are not sequential, so any detected links reflect co-variation rather than temporal causality. This is merely an illustrative example to show the application.

In [ ]:
wine = load_wine()
X_wine, y_wine = wine.data, wine.target
feature_names = list(wine.feature_names)

mi_scores = mutual_info_classif(X_wine, y_wine, random_state=SEED)
order = np.argsort(mi_scores)[::-1]
f0, f1 = feature_names[order[0]], feature_names[order[1]]
print(f"Top-2 features: '{f0}' and '{f1}'")

# Discretise to 5 bins
def discretize_1d(arr, n_bins=5):
    est = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='uniform')
    return est.fit_transform(arr.reshape(-1, 1)).ravel().astype(int).tolist()

a = discretize_1d(X_wine[:, order[0]])
b = discretize_1d(X_wine[:, order[1]])

te_ab = transfer_entropy(a, b, k=1)
te_ba = transfer_entropy(b, a, k=1)
ais_a = active_info(a, k=1)
ais_b = active_info(b, k=1)

print(f"\nTransfer Entropy (Wine, treating samples as a sequence)")
print(f"  TE({f0[:12]}→{f1[:12]}) = {te_ab:.4f} bits")
print(f"  TE({f1[:12]}→{f0[:12]}) = {te_ba:.4f} bits")
print(f"\nActive Information Storage")
print(f"  AIS({f0[:12]}) = {ais_a:.4f} bits")
print(f"  AIS({f1[:12]}) = {ais_b:.4f} bits")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar([f'TE({f0[:10]}→{f1[:10]})', f'TE({f1[:10]}→{f0[:10]})'],
            [te_ab, te_ba],
            color=[sns.color_palette()[0], sns.color_palette()[3]])
axes[0].set_ylabel('Transfer Entropy (bits)')
axes[0].set_title('Wine: directed TE between top-2 features', fontweight='bold')

axes[1].bar([f'AIS({f0[:10]})', f'AIS({f1[:10]})'], [ais_a, ais_b],
            color=[sns.color_palette()[2], sns.color_palette()[1]])
axes[1].set_ylabel('Active Information Storage (bits)')
axes[1].set_title('Wine: AIS of top-2 features', fontweight='bold')

plt.tight_layout()
plt.show()
